[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_06/ATS_ch6_ucsv_inflation/ATS_ch6_ucsv_inflation.ipynb)

# ATS_ch6_ucsv_inflation

Trend inflation with stochastic volatility (UC-SV, Stock and Watson 2007): US GDP-deflator inflation since 1953 and Romanian HICP inflation since 2001 (with quarterly seasonal dummies); Gibbs sampler with the precision sampler and the KSC mixture; the BNR target band.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 6: State space models and Bayesian filtering. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_6'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import time

from scipy import linalg

SEED = 2026

LOG2PI = np.log(2 * np.pi)

KSC_P = np.array([0.0073, 0.10556, 2e-05, 0.04395, 0.34001, 0.24566, 0.2575])

KSC_M = np.array([-10.12999, -3.97281, -8.56686, 2.77786, 0.61942, 1.79518, -1.08819])

KSC_V = np.array([5.79596, 2.61369, 5.1795, 0.16735, 0.64009, 0.34023, 1.26261])

SV = {'start': '2016-01-01', 'end': '2026-09-18', 'offset': 0.001}

MNZ = {'start': '1947-01-01', 'end': '1998-04-01'}

UCSV = {'gamma': 0.2, 'start_us': '1953-01-01', 'start_ro': '2001-01-01'}

TVP = {'start': '2005-08-01'}

DFM_SERIES = {'INDPRO': 'industrial production', 'PAYEMS': 'payroll employment', 'W875RX1': 'real personal income less transfers', 'CMRMTSPL': 'real manufacturing and trade sales'}

_sv_cache = {}

def save(name, save_it=True):
    st.check_no_grey(plt.gcf())
    if save_it:
        st.save_fig(name)
    else:
        plt.show()
        plt.close()


def ssm(Z, T, R, Q, H, a1=None, P1=None, Pinf=None, c=None):
    """A time-invariant state space model (Durbin and Koopman 2012 notation), univariate observation:
        y_t = Z_t alpha_t + eps_t,          eps_t ~ N(0, H)
        alpha_{t+1} = c + T alpha_t + R eta_t,   eta_t ~ N(0, Q)
        alpha_1 ~ N(a1, P1 + kappa * Pinf), kappa -> infinity (exact diffuse part Pinf).
    Z may be a vector (m,) or a matrix (n, m) of time-varying rows (regressions)."""
    T = np.atleast_2d(np.asarray(T, float))
    m = T.shape[0]
    R = np.atleast_2d(np.asarray(R, float)).reshape(m, -1)
    Q = np.atleast_2d(np.asarray(Q, float))
    return dict(Z=np.asarray(Z, float), T=T, R=R, Q=Q, RQR=R @ Q @ R.T, H=float(H),
                a1=np.zeros(m) if a1 is None else np.asarray(a1, float),
                P1=np.zeros((m, m)) if P1 is None else np.atleast_2d(np.asarray(P1, float)),
                Pinf=np.zeros((m, m)) if Pinf is None else np.atleast_2d(np.asarray(Pinf, float)),
                c=np.zeros(m) if c is None else np.asarray(c, float))


def _zrow(mod, t):
    Z = mod['Z']
    return Z[t] if Z.ndim == 2 else Z


def kalman_filter(y, mod, tol=1e-8):
    """Kalman filter with exact diffuse initialisation (Koopman 1997; Durbin and Koopman 2012, Section 5.2) for a
    univariate observation. Missing values (NaN) skip the update. Returns predicted states a_t, P_t (P_*), the
    diffuse parts, innovations v_t, variances F_t and the diffuse log-likelihood (Durbin and Koopman 2012, eq. 7.4)."""
    y = np.asarray(y, float)
    n, m = len(y), mod['T'].shape[0]
    T, RQR, H, c = mod['T'], mod['RQR'], mod['H'], mod['c']
    a, P, Pi = mod['a1'].copy(), mod['P1'].copy(), mod['Pinf'].copy()
    out = dict(a=np.zeros((n, m)), P=np.zeros((n, m, m)), Pinf=np.zeros((n, m, m)), v=np.full(n, np.nan), att=np.zeros((n, m)),
               F=np.full(n, np.nan), Finf=np.zeros(n), K0=np.zeros((n, m)), K1=np.zeros((n, m)),
               ll=np.zeros(n), diffuse=np.zeros(n, bool))
    d = 0
    for t in range(n):
        Z = _zrow(mod, t)
        out['a'][t], out['P'][t], out['Pinf'][t] = a, P, Pi
        is_diff = np.abs(Pi).max() > tol
        out['diffuse'][t] = is_diff
        if np.isnan(y[t]):                       # missing observation: prediction only
            out['att'][t] = a
            a, P, Pi = c + T @ a, T @ P @ T.T + RQR, T @ Pi @ T.T
            continue
        v = y[t] - Z @ a
        Ms, Fs = P @ Z, Z @ P @ Z + H
        out['v'][t] = v
        if is_diff:
            Mi = Pi @ Z
            Fi = Z @ Mi
            out['Finf'][t] = Fi
            if Fi > tol:
                out['att'][t] = a + Mi * v / Fi
                F1, F2 = 1 / Fi, -Fs / Fi ** 2
                K0, K1 = T @ Mi * F1, T @ (Ms * F1 + Mi * F2)
                L0, L1 = T - np.outer(K0, Z), -np.outer(K1, Z)
                a = c + T @ a + K0 * v
                Pi, P = T @ Pi @ L0.T, T @ Pi @ L1.T + T @ P @ L0.T + RQR
                out['F'][t], out['K0'][t], out['K1'][t] = Fi, K0, K1
                out['ll'][t] = -0.5 * (LOG2PI + np.log(Fi))
            else:
                out['att'][t] = a + Ms * v / Fs
                K0 = T @ Ms / Fs
                L0 = T - np.outer(K0, Z)
                a = c + T @ a + K0 * v
                Pi, P = T @ Pi @ T.T, T @ P @ L0.T + RQR
                out['F'][t], out['K0'][t] = Fs, K0
                out['ll'][t] = -0.5 * (LOG2PI + np.log(Fs) + v ** 2 / Fs)
            d = t + 1
        else:
            out['att'][t] = a + Ms * v / Fs
            K = T @ Ms / Fs
            a = c + T @ a + K * v
            P = T @ P @ (T - np.outer(K, Z)).T + RQR
            out['F'][t], out['K0'][t] = Fs, K
            out['ll'][t] = -0.5 * (LOG2PI + np.log(Fs) + v ** 2 / Fs)
        P = 0.5 * (P + P.T)
    out['d'], out['loglik'] = d, float(out['ll'].sum())
    out['a_next'], out['P_next'] = a, P
    return out


def kalman_smoother(y, mod, kf=None, tol=1e-8):
    """State smoother with exact diffuse initialisation (Durbin and Koopman 2012, Section 5.3): backward recursions
    for r_t, N_t (and r^(1), N^(1), N^(2) in the diffuse period). Returns smoothed states and their variances."""
    y = np.asarray(y, float)
    kf = kf or kalman_filter(y, mod, tol)
    n, m = len(y), mod['T'].shape[0]
    T, H = mod['T'], mod['H']
    r0, r1 = np.zeros(m), np.zeros(m)
    N0, N1, N2 = np.zeros((m, m)), np.zeros((m, m)), np.zeros((m, m))
    ah, V = np.zeros((n, m)), np.zeros((n, m, m))
    for t in range(n - 1, -1, -1):
        Z = _zrow(mod, t)
        a, P, Pi, v = kf['a'][t], kf['P'][t], kf['Pinf'][t], kf['v'][t]
        if np.isnan(y[t]):
            r0, N0 = T.T @ r0, T.T @ N0 @ T
            if kf['diffuse'][t]:
                r1, N1, N2 = T.T @ r1, T.T @ N1 @ T, T.T @ N2 @ T
        elif not kf['diffuse'][t]:
            F = kf['F'][t]
            L = T - np.outer(kf['K0'][t], Z)
            r0 = Z * v / F + L.T @ r0
            N0 = np.outer(Z, Z) / F + L.T @ N0 @ L
        else:
            Fi = kf['Finf'][t]
            if Fi > tol:
                Fs = Z @ P @ Z + H
                F1, F2 = 1 / Fi, -Fs / Fi ** 2
                L0, L1 = T - np.outer(kf['K0'][t], Z), -np.outer(kf['K1'][t], Z)
                r1n = Z * v * F1 + L0.T @ r1 + L1.T @ r0
                r0n = L0.T @ r0
                N2n = (np.outer(Z, Z) * F2 + L0.T @ N2 @ L0 + L0.T @ N1 @ L1 + L1.T @ N1 @ L0
                       + L1.T @ N0 @ L1)
                N1n = np.outer(Z, Z) * F1 + L0.T @ N1 @ L0 + L1.T @ N0 @ L0 + L0.T @ N0 @ L1
                N0n = L0.T @ N0 @ L0
                r0, r1, N0, N1, N2 = r0n, r1n, N0n, N1n, N2n
            else:
                Fs = kf['F'][t]
                L0 = T - np.outer(kf['K0'][t], Z)
                r0 = Z * v / Fs + L0.T @ r0
                N0 = np.outer(Z, Z) / Fs + L0.T @ N0 @ L0
                r1, N1, N2 = T.T @ r1, T.T @ N1 @ L0, T.T @ N2 @ T
        if kf['diffuse'][t]:
            ah[t] = a + P @ r0 + Pi @ r1
            V[t] = P - P @ N0 @ P - (Pi @ N1 @ P).T - Pi @ N1 @ P - Pi @ N2 @ Pi
        else:
            ah[t] = a + P @ r0
            V[t] = P - P @ N0 @ P
    return dict(alpha=ah, V=V, kf=kf)


def big_kappa(mod, kappa):
    """The same model with the diffuse part replaced by kappa * Pinf (the approximate diffuse initialisation)."""
    m2 = dict(mod)
    m2['P1'] = mod['P1'] + kappa * mod['Pinf']
    m2['Pinf'] = np.zeros_like(mod['Pinf'])
    return m2


def local_level(s2eps, s2eta):
    """Local level model: y_t = mu_t + eps_t, mu_{t+1} = mu_t + eta_t, mu_1 diffuse."""
    return ssm([1.0], [[1.0]], [[1.0]], [[s2eta]], s2eps, Pinf=[[1.0]])


def num_hessian(f, x, h=1e-4):
    x = np.asarray(x, float)
    k = len(x)
    Hm = np.zeros((k, k))
    for i in range(k):
        for j in range(i, k):
            e1, e2 = np.zeros(k), np.zeros(k)
            e1[i], e2[j] = h, h
            Hm[i, j] = Hm[j, i] = (f(x + e1 + e2) - f(x + e1 - e2) - f(x - e1 + e2) + f(x - e1 - e2)) / (4 * h * h)
    return Hm


def fit_local_level(y):
    """Exact diffuse ML of the local level model with numpy: log-variances, BFGS from a grid start."""
    y = np.asarray(y, float)
    f = lambda th: -kalman_filter(y, local_level(np.exp(th[0]), np.exp(th[1])))['loglik']   # noqa: E731
    v = np.nanvar(np.diff(y))
    starts = [np.log([v * a, v * (1 - a) + 1e-6]) for a in (0.2, 0.5, 0.8)]
    best = min((optimize.minimize(f, s, method='BFGS') for s in starts), key=lambda r: r.fun)
    th = best.x
    Hn = num_hessian(f, th)
    se = np.sqrt(np.diag(np.linalg.inv(Hn)))
    return dict(s2eps=float(np.exp(th[0])), s2eta=float(np.exp(th[1])), loglik=float(-best.fun), se_log=se.tolist(),
                q=float(np.exp(th[1] - th[0])))


def us_inflation(start='1953-01-01', end=None):
    """US quarterly inflation, GDP price deflator (FRED GDPDEF), 400 x log difference (% a.r.)."""
    p = read_fred('GDPDEF')
    pi = 400 * np.log(p).diff().dropna()
    return pi.loc[start:end].rename('pi_us')


def hicp_index(geo):
    """Monthly HICP, all items, index 2025 = 100 (Eurostat prc_hicp_minr)."""
    return read_eurostat('prc_hicp_minr', f'M.I25.TOTAL.{geo}')


def hicp_yoy(geo):
    """12-month HICP inflation, % (Eurostat prc_hicp_minr, annual rate of change)."""
    return read_eurostat('prc_hicp_minr', f'M.RCH_A.TOTAL.{geo}')


def ro_quarterly_inflation(start=UCSV['start_ro']):
    """Romanian quarterly HICP inflation: quarterly averages of the monthly index, 400 x log difference (% a.r.),
    complete quarters only."""
    p = hicp_index('RO')
    q = p.resample('QS').mean()
    cnt = p.resample('QS').count()
    q = q[cnt == 3]
    return (400 * np.log(q).diff()).dropna().loc[start:].rename('pi_ro')


def gdp_log(geo='RO', start='1995-01-01'):
    """100 x log real GDP, quarterly, seasonally and calendar adjusted, chain-linked volumes 2010 (Eurostat
    namq_10_gdp); for 'US' the FRED series GDPC1."""
    if geo == 'US':
        s = read_fred('GDPC1')
    else:
        s = read_eurostat('namq_10_gdp', f'Q.CLV10_MEUR.SCA.B1GQ.{geo}')
    return (100 * np.log(s)).loc[start:].rename(f'y_{geo}')


def returns(name, start=SV['start'], end=SV['end']):
    """Daily log returns in % (course data, EODHD), demeaned."""
    r = 100 * np.log(load_close(name)).diff().dropna().loc[start:end]
    return r - r.mean()


def precision_rw(y, s2eps, s2eta, m0=0.0, v0=1e6, rng=None, draw=True):
    """Precision sampler (Chan and Jeliazkov 2009) for y_t = mu_t + eps_t, mu_t = mu_{t-1} + eta_t, mu_1 ~ N(m0, v0):
    the posterior precision of mu is tridiagonal, so one banded Cholesky factorisation gives the mean and a draw in
    O(n). s2eps and s2eta may be vectors (time-varying variances)."""
    y = np.asarray(y, float)
    n = len(y)
    se = np.broadcast_to(np.asarray(s2eps, float), (n,))
    sh = np.broadcast_to(np.asarray(s2eta, float), (n,)).copy()
    w = 1 / sh                                    # w[t] = 1 / var(mu_t - mu_{t-1}) for t >= 1 (w[0] unused)
    diag = 1 / se + np.r_[1 / v0, np.zeros(n - 1)] + np.r_[w[1:], 0] + np.r_[0, w[1:]]
    off = -w[1:]
    ab = np.zeros((2, n))
    ab[0, 1:], ab[1] = off, diag                 # upper banded form for cholesky_banded
    U = linalg.cholesky_banded(ab, lower=False)
    b = y / se + np.r_[m0 / v0, np.zeros(n - 1)]
    mean = linalg.cho_solve_banded((U, False), b)
    if not draw:
        return mean
    rng = rng or np.random.default_rng()
    return mean + linalg.solve_banded((0, 1), U, rng.standard_normal(n))


def draw_s_ksc(ystar, h, rng):
    """Mixture indicators: P(s_t = i) proportional to p_i N(y*_t - h_t; m_i - 1.2704, v_i)."""
    e = (ystar - h)[:, None] - (KSC_M - 1.2704)[None, :]
    lw = np.log(KSC_P)[None, :] - 0.5 * np.log(KSC_V)[None, :] - 0.5 * e ** 2 / KSC_V[None, :]
    w = np.exp(lw - lw.max(1, keepdims=True))
    cw = np.cumsum(w / w.sum(1, keepdims=True), axis=1)
    return (rng.random(len(h))[:, None] > cw).sum(1).clip(0, 6)


def ucsv_gibbs(pi, gamma=UCSV['gamma'], draws=20000, burn=5000, seasonal=None, rng=None, offset=1e-3):
    """UC-SV of Stock and Watson (2007): pi_t = tau_t + exp(h1_t/2) eta_t, tau_t = tau_{t-1} + exp(h2_t/2) eps_t,
    h_jt = h_j,t-1 + nu_jt, var(nu) = gamma (fixed). Gibbs: tau by the precision sampler; the log-volatilities by the
    KSC mixture and the precision sampler; optional quarterly seasonal dummies (sum-to-zero) for a non-adjusted series."""
    rng = rng or np.random.default_rng(SEED)
    pi = np.asarray(pi, float)
    n = len(pi)
    D = None
    if seasonal is not None:
        qq = np.asarray(seasonal)
        D = np.column_stack([(qq == k).astype(float) - (qq == 4).astype(float) for k in (1, 2, 3)])
    g = np.zeros(3)
    v0 = np.log(np.var(pi))
    h1, h2 = np.full(n, v0), np.full(n - 1, v0 - 2)
    tau = pi.copy()
    keep = dict(tau=np.zeros((draws, n)), s1=np.zeros(n), s2=np.zeros(n - 1), g=np.zeros((draws, 3)))
    for it in range(draws + burn):
        ys = pi - (D @ g if D is not None else 0)
        tau = precision_rw(ys, np.exp(h1), np.r_[1.0, np.exp(h2)], ys[:4].mean(), 25.0, rng)
        e1 = ys - tau
        e2 = np.diff(tau)
        for e, which in ((e1, 1), (e2, 2)):
            yst = np.log(e ** 2 + offset)
            h = h1 if which == 1 else h2
            s = draw_s_ksc(yst, h, rng)
            h = precision_rw(yst - (KSC_M[s] - 1.2704), KSC_V[s], gamma, h[0], 10.0, rng)
            if which == 1:
                h1 = h
            else:
                h2 = h
        if D is not None:
            w = np.exp(-h1)
            Vp = np.linalg.inv(D.T @ (D * w[:, None]) + np.eye(3) / 10)
            g = Vp @ (D.T @ (w * (pi - tau))) + np.linalg.cholesky(Vp) @ rng.standard_normal(3)
        if it >= burn:
            k = it - burn
            keep['tau'][k] = tau
            keep['s1'] += np.exp(h1 / 2)
            keep['s2'] += np.exp(h2 / 2)
            keep['g'][k] = g
    keep['s1'] /= draws
    keep['s2'] /= draws
    return keep


def fig_ucsv_us(save_it=True, draws=20000, burn=5000):
    """UC-SV trend of US GDP-deflator inflation, 1953Q1 onward: trend with 68% band; permanent and transitory
    volatilities; the implied IMA(1,1) coefficient theta_t (Stock and Watson 2007, Fig. 4 logic)."""
    pi = us_inflation(UCSV['start_us'])
    k = ucsv_gibbs(pi.values, draws=draws, burn=burn)
    tau = k['tau']
    lo, md, hi = np.quantile(tau, [0.16, 0.5, 0.84], axis=0)
    s1, s2 = k['s1'], np.r_[np.nan, k['s2']]
    q = (s2 / s1) ** 2
    theta = -(1 + q / 2 - np.sqrt(q + q ** 2 / 4))
    fig, ax = plt.subplots(2, 1, figsize=(11, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[1.4, 1]))
    ax[0].plot(pi.index, pi.values, color=st.Amber, lw=0.8, label='inflation, GDP deflator (% a.r.)')
    ax[0].plot(pi.index, md, color=st.MainBlue, lw=1.7, label='UC-SV trend (posterior median)')
    ax[0].fill_between(pi.index, lo, hi, color=st.MainBlue, alpha=0.2, label='68% band')
    ax[0].set_ylabel('% a.r.')
    ax[1].plot(pi.index, s2, color=st.IDAred, lw=1.4, label='s.d. of trend shocks (permanent)')
    ax[1].plot(pi.index, s1, color=st.Forest, lw=1.4, label='s.d. of transitory shocks')
    ax[1].set_ylabel('posterior mean')
    st.fig_legend_bottom(fig, ncol=3, y=-0.0)
    fig.tight_layout()
    save('ats_ch6_ucsv_us', save_it)
    at = {k_: int(np.argmin(np.abs(pi.index - pd.Timestamp(k_)))) for k_ in
          ('1975-01-01', '1980-01-01', '1995-01-01', '2005-01-01', '2019-10-01', '2022-04-01')}
    return dict(T=len(pi), first=str(pi.index[0].date()), last=str(pi.index[-1].date()), draws=draws, burn=burn,
                tau_at={k_: [float(lo[i]), float(md[i]), float(hi[i])] for k_, i in at.items()},
                s1_at={k_: float(s1[i]) for k_, i in at.items()}, s2_at={k_: float(s2[i]) for k_, i in at.items()},
                theta_at={k_: float(theta[i]) for k_, i in at.items()},
                tau_last=[float(lo[-1]), float(md[-1]), float(hi[-1])], pi_last=float(pi.values[-1]),
                pi_4q=float(pi.values[-4:].mean()))


def fig_ucsv_ro(save_it=True, draws=20000, burn=5000):
    """UC-SV trend of Romanian HICP inflation, quarterly 2001Q1 onward, with quarterly seasonal dummies; the BNR
    target band (2.5% +/- 1 pp since 2013)."""
    pi = ro_quarterly_inflation()
    k = ucsv_gibbs(pi.values, draws=draws, burn=burn, seasonal=pi.index.quarter)
    tau = k['tau']
    lo, md, hi = np.quantile(tau, [0.16, 0.5, 0.84], axis=0)
    fig, ax = plt.subplots(figsize=(11, 3.9))
    sa = pi.values - (np.column_stack([(pi.index.quarter == j).astype(float) - (pi.index.quarter == 4).astype(float)
                                       for j in (1, 2, 3)]) @ k['g'].mean(0))
    ax.plot(pi.index, sa, color=st.Amber, lw=0.8, label='HICP inflation net of seasonal effects (% a.r.)')
    ax.plot(pi.index, md, color=st.MainBlue, lw=1.7, label='UC-SV trend (posterior median)')
    ax.fill_between(pi.index, lo, hi, color=st.MainBlue, alpha=0.2, label='68% band')
    tgt = pi.index >= pd.Timestamp('2013-01-01')
    ax.fill_between(pi.index[tgt], 1.5, 3.5, color=st.Forest, alpha=0.12, label='BNR target band 2.5% +/- 1 pp')
    ax.set_ylabel('% a.r.')
    ax.set_ylim(-5, max(25, np.nanmax(sa[pi.index >= '2003-01-01']) + 1))
    st.legend_outside_bottom(ax, ncol=2, y=-0.15)
    save('ats_ch6_ucsv_ro', save_it)
    post_in = float(np.mean((tau[:, -1] >= 1.5) & (tau[:, -1] <= 3.5)))
    at = {k_: int(np.argmin(np.abs(pi.index - pd.Timestamp(k_)))) for k_ in
          ('2005-07-01', '2015-07-01', '2019-10-01', '2023-01-01')}
    return dict(T=len(pi), first=str(pi.index[0].date()), last=str(pi.index[-1].date()), draws=draws,
                tau_at={k_: [float(lo[i]), float(md[i]), float(hi[i])] for k_, i in at.items()},
                tau_last=[float(lo[-1]), float(md[-1]), float(hi[-1])], post_in_band=post_in,
                seas=k['g'].mean(0).tolist(), s1_last=float(k['s1'][-1]), s2_last=float(k['s2'][-1]))

## Run

In [ ]:
print(fig_ucsv_us(draws=4000, burn=1000))
print(fig_ucsv_ro(draws=4000, burn=1000))

![ats_ch6_ucsv_us](./ats_ch6_ucsv_us.png)

Output: `ats_ch6_ucsv_us.pdf`

![ats_ch6_ucsv_ro](./ats_ch6_ucsv_ro.png)

Output: `ats_ch6_ucsv_ro.pdf`